# 02 — Ingestion: NHS England ADHD Management Information

**Project:** ADHD Care Equity Tracker UK
**Notebook purpose:** Download and persist the NHS England ADHD Management Information (MI-ADHD) publication — the most ADHD-specific national waiting-times dataset published by any UK statistics body. First released 29 May 2025 as part of NHS England's ADHD data improvement plan; this notebook ingests the **February 2026** release.
**Author:** Noble Chidera Onyema
**Created:** 17 May 2026

---

© 2026 Noble Chidera Onyema. All Rights Reserved.
See `LICENSE` and `NOTICE.md` in the repository root. No commercial use, derivative works, redistribution, or use as ML training data is permitted without written permission.

In [1]:
"""
02_mi_adhd_ingestion.ipynb — NHS England MI-ADHD download and validation.

Copyright (c) 2026 Noble Chidera Onyema. All Rights Reserved.
"""

from pathlib import Path
import sys
import requests
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

assert DATA_RAW.exists(), f"data/raw not found at {DATA_RAW}"
assert DATA_PROCESSED.exists(), f"data/processed not found at {DATA_PROCESSED}"

print(f"Python:    {sys.version.split()[0]}")
print(f"pandas:    {pd.__version__}")
print(f"requests:  {requests.__version__}")
print(f"Raw dir:   {DATA_RAW}")

Python:    3.11.9
pandas:    2.2.3
requests:  2.32.3
Raw dir:   C:\Users\HP\Projects\adhd-care-equity-tracker\data\raw


In [2]:
# openpyxl is required to read the .xlsx data dictionary published by NHS England.
# Pinned for reproducibility; also added to requirements.txt.
%pip install openpyxl==3.1.5 --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# Three source files from the MI-ADHD February 2026 publication on
# digital.nhs.uk. URLs verified live on 16 May 2026.
FILES = {
    "mi_adhd_feb2026_main.csv":          "https://files.digital.nhs.uk/36/B23DC8/ADHD_Feb26.csv",
    "mi_adhd_feb2026_summary.xlsx":      "https://files.digital.nhs.uk/B8/2C3AD4/ADHD_summary_Feb26.xlsx",
    "mi_adhd_data_dictionary_v1.2.xlsx": "https://files.digital.nhs.uk/B0/2C9963/Data_dictionary_v1.2.xlsx",
}

for local_name, url in FILES.items():
    target = DATA_RAW / local_name
    if target.exists():
        print(f"{local_name:42s}  cached  {target.stat().st_size / 1024:.1f} KB")
        continue
    with requests.get(url, headers=HEADERS, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(target, "wb") as f:
            for chunk in r.iter_content(chunk_size=64 * 1024):
                f.write(chunk)
    print(f"{local_name:42s}  fetched {target.stat().st_size / 1024:.1f} KB")

mi_adhd_feb2026_main.csv                    cached  513.8 KB
mi_adhd_feb2026_summary.xlsx                cached  115.5 KB
mi_adhd_data_dictionary_v1.2.xlsx           cached  36.0 KB


## Data dictionary

NHS England ships a separate workbook (`Data_dictionary_v1.2.xlsx`) defining every indicator code used in the main CSV. We inspect it first so the rest of the notebook works with documented definitions rather than guessed-at column meanings.

In [4]:
DICT_PATH = DATA_RAW / "mi_adhd_data_dictionary_v1.2.xlsx"

xls = pd.ExcelFile(DICT_PATH)
print(f"Sheets in dictionary: {xls.sheet_names}\n")

for sheet in xls.sheet_names:
    df = pd.read_excel(DICT_PATH, sheet_name=sheet)
    print(f"--- {sheet} ---")
    print(f"  shape:   {df.shape}")
    print(f"  columns: {df.columns.tolist()}")
    print()

Sheets in dictionary: ['Title page', 'Data dictionary']

--- Title page ---
  shape:   (40, 1)
  columns: ['Unnamed: 0']

--- Data dictionary ---
  shape:   (28, 6)
  columns: ['Metric ID', 'Name', 'Description', 'Geog Breakdown', 'Age Breakdowns', 'Other Breakdowns']



In [5]:
pd.set_option("display.max_colwidth", None)
dict_df = pd.read_excel(DICT_PATH, sheet_name="Data dictionary")

print(f"Indicators defined: {dict_df.shape[0]}\n")
for _, row in dict_df.iterrows():
    print(f"{row['Metric ID']:10s}  {row['Name']}")
    if pd.notna(row['Description']):
        print(f"             {row['Description']}")
    print()

Indicators defined: 28

ADHD001     Estimated ADHD prevalence
             An estimate of the number of people with ADHD in England (including those without a formal diagnosis) calculated using GP list size data and NICE prevalence estimates.

ADHD003     The number of open referrals that may be for an ADHD assessment
             The number of open referrals that may be for an ADHD assessment as recorded in MHSDS, including those who have had a first contact.

ADHD003a    The number of open referrals that may be for an ADHD assessment that have been open for up to 13 weeks

ADHD003b    The number of open referrals that may be for an ADHD assessment that have been open for between 13 and up to 52 weeks

ADHD003c    The number of open referrals that may be for an ADHD assessment that have been open for between 52 and up to 104 weeks

ADHD003d    The number of open referrals that may be for an ADHD assessment that have been open for 104 weeks or more

ADHD004     The number of open refer

## Main data file

The main CSV is a long-format table with one row per indicator per breakdown level per reporting period. NHS England suppresses small-count cells with `*` for statistical disclosure control; we coerce `VALUE` to numeric on read and surface the count of suppressed cells.

In [6]:
MAIN_PATH = DATA_RAW / "mi_adhd_feb2026_main.csv"

mi_adhd = pd.read_csv(MAIN_PATH, low_memory=False)
print(f"Shape: {mi_adhd.shape}")
print(f"\nColumns ({len(mi_adhd.columns)}):")
for col in mi_adhd.columns:
    print(f"  - {col}")

# NHS England suppresses small-count cells with '*' for disclosure control.
# Coerce VALUE to numeric; suppressed cells become NaN.
non_numeric = mi_adhd.loc[
    pd.to_numeric(mi_adhd["VALUE"], errors="coerce").isna(), "VALUE"
].unique()
print(f"\nNon-numeric VALUE entries found: {non_numeric}")

mi_adhd["VALUE"] = pd.to_numeric(mi_adhd["VALUE"], errors="coerce")
print(f"Suppressed (NaN) cells: {mi_adhd['VALUE'].isna().sum()}")
print(f"VALUE dtype: {mi_adhd['VALUE'].dtype}")

print(f"\nMemory: {mi_adhd.memory_usage(deep=True).sum() / 1024:.1f} KB")
mi_adhd.head()

Shape: (8166, 7)

Columns (7):
  - REPORTING_PERIOD_START_DATE
  - REPORTING_PERIOD_END_DATE
  - BREAKDOWN
  - PRIMARY_LEVEL
  - PRIMARY_LEVEL_DESCRIPTION
  - INDICATOR_ID
  - VALUE

Non-numeric VALUE entries found: ['*']
Suppressed (NaN) cells: 240
VALUE dtype: float64

Memory: 3214.4 KB


,REPORTING_PERIOD_START_DATE,REPORTING_PERIOD_END_DATE,BREAKDOWN,PRIMARY_LEVEL,PRIMARY_LEVEL_DESCRIPTION,INDICATOR_ID,VALUE
0,01/11/2025,01/11/2025,Age Group,0 to 4,People aged 0 to 4,ADHD001,146000.0
1,01/11/2025,01/11/2025,Age Group,5 to 17,People aged 5 to 17,ADHD001,475000.0
2,01/11/2025,01/11/2025,Age Group,18 to 24,People aged 18 to 24,ADHD001,269000.0
3,01/11/2025,01/11/2025,Age Group,25+,People aged 25+,ADHD001,1616000.0
4,2025-02-01,2025-02-01,Age Group,0 to 4,People aged 0 to 4,ADHD001,148000.0


In [7]:
print(f"Distinct INDICATOR_ID values ({mi_adhd['INDICATOR_ID'].nunique()}):")
for ind in sorted(mi_adhd['INDICATOR_ID'].unique()):
    n_rows = (mi_adhd['INDICATOR_ID'] == ind).sum()
    print(f"  {ind}  ({n_rows:,} rows)")

print(f"\nDistinct BREAKDOWN values ({mi_adhd['BREAKDOWN'].nunique()}):")
for b in sorted(mi_adhd['BREAKDOWN'].unique()):
    n_rows = (mi_adhd['BREAKDOWN'] == b).sum()
    print(f"  {b}  ({n_rows:,} rows)")

print(f"\nRaw date values (first 10): {mi_adhd['REPORTING_PERIOD_START_DATE'].head(10).tolist()}")

Distinct INDICATOR_ID values (27):
  ADHD001  (24 rows)
  ADHD003  (390 rows)
  ADHD003a  (387 rows)
  ADHD003b  (388 rows)
  ADHD003c  (390 rows)
  ADHD003d  (387 rows)
  ADHD004  (390 rows)
  ADHD004a  (387 rows)
  ADHD004b  (388 rows)
  ADHD004c  (390 rows)
  ADHD004d  (387 rows)
  ADHD005  (390 rows)
  ADHD005a  (377 rows)
  ADHD005b  (377 rows)
  ADHD005c  (377 rows)
  ADHD005d  (377 rows)
  ADHD006  (390 rows)
  ADHD006a  (378 rows)
  ADHD006b  (377 rows)
  ADHD006c  (383 rows)
  ADHD006d  (377 rows)
  ADHD007  (390 rows)
  ADHD008  (13 rows)
  ADHD008a  (13 rows)
  ADHD008b  (13 rows)
  ADHD008c  (13 rows)
  ADHD008d  (13 rows)

Distinct BREAKDOWN values (3):
  Age Group  (1,344 rows)
  Ethnicity  (5,187 rows)
  Gender  (1,635 rows)

Raw date values (first 10): ['01/11/2025', '01/11/2025', '01/11/2025', '01/11/2025', '2025-02-01', '2025-02-01', '2025-02-01', '2025-02-01', '2025-05-01', '2025-05-01']


parquet_path = DATA_PROCESSED / "mi_adhd_feb2026.parquet"
mi_adhd.to_parquet(parquet_path, engine="pyarrow", compression="snappy", index=False)
print(f"Saved: {parquet_path.name}  ({parquet_path.stat().st_size / 1024:.1f} KB)")

## Date parsing

The published CSV mixes two date formats in the same column: UK `DD/MM/YYYY` for some rows and ISO `YYYY-MM-DD` for others. We parse both and verify zero rows fail.

In [8]:
def parse_mixed_date(value):
    """Try UK DD/MM/YYYY first, fall back to ISO YYYY-MM-DD."""
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

mi_adhd["REPORTING_PERIOD_START_DATE"] = mi_adhd["REPORTING_PERIOD_START_DATE"].apply(parse_mixed_date)
mi_adhd["REPORTING_PERIOD_END_DATE"]   = mi_adhd["REPORTING_PERIOD_END_DATE"].apply(parse_mixed_date)

n_failed = mi_adhd["REPORTING_PERIOD_START_DATE"].isna().sum()
print(f"Dates parsed. Rows that failed parsing: {n_failed}")
print(f"Date range: {mi_adhd['REPORTING_PERIOD_START_DATE'].min().date()} to {mi_adhd['REPORTING_PERIOD_END_DATE'].max().date()}")

Dates parsed. Rows that failed parsing: 0
Date range: 2024-12-01 to 2026-02-01


## Validation against the official figure

We cast string columns to categorical (memory win), then test our load by reproducing the headline figure published by NHS England for December 2025: **562,480 open referrals that may be for an ADHD assessment** (Mental Health Services Dataset only). If our sum matches to the unit, the data is loaded correctly.

Note on reporting cadence: the prevalence series (ADHD001) is estimated more frequently than the activity series (ADHD003+), which depend on provider submissions to MHSDS and typically lag by two months. The latest period varies by indicator family.

In [9]:
for col in ["BREAKDOWN", "PRIMARY_LEVEL", "PRIMARY_LEVEL_DESCRIPTION", "INDICATOR_ID"]:
    mi_adhd[col] = mi_adhd[col].astype("category")

print(f"Memory after categorisation: {mi_adhd.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Shape: {mi_adhd.shape}\n")

# Latest reporting period varies by indicator family.
print("Latest reporting period by indicator family:")
for ind in ["ADHD001", "ADHD003", "ADHD006", "ADHD007", "ADHD008"]:
    sub = mi_adhd[mi_adhd["INDICATOR_ID"] == ind]
    if not sub.empty:
        latest = sub["REPORTING_PERIOD_START_DATE"].max()
        print(f"  {ind}: latest = {latest.date()}")

# Validate against the official NHS England figure for December 2025.
DATA_LATEST = pd.Timestamp("2025-12-01")
age_open = mi_adhd[
    (mi_adhd["REPORTING_PERIOD_START_DATE"] == DATA_LATEST)
    & (mi_adhd["BREAKDOWN"] == "Age Group")
    & (mi_adhd["INDICATOR_ID"] == "ADHD003")
]
total = age_open["VALUE"].sum()
print(f"\nADHD003 open referrals at {DATA_LATEST.date()}, by age group:")
print(age_open[["PRIMARY_LEVEL_DESCRIPTION", "VALUE"]].to_string(index=False))
print(f"\nTotal: {total:,.0f}")
print(f"Official NHS England figure for December 2025: 562,480")
print(f"Match: {abs(total - 562480) < 1}")

Memory after categorisation: 232.3 KB
Shape: (8166, 7)

Latest reporting period by indicator family:
  ADHD001: latest = 2026-02-01
  ADHD003: latest = 2025-12-01
  ADHD006: latest = 2025-12-01
  ADHD007: latest = 2025-12-01
  ADHD008: latest = 2025-12-01

ADHD003 open referrals at 2025-12-01, by age group:
PRIMARY_LEVEL_DESCRIPTION    VALUE
          People aged 25+ 292425.0
      People aged 5 to 17 161885.0
      People aged Unknown     30.0
       People aged 0 to 4   3310.0
     People aged 18 to 24 104830.0

Total: 562,480
Official NHS England figure for December 2025: 562,480
Match: True


## Persist as parquet

Final cleaned dataframe is saved as a snappy-compressed parquet file in `data/processed/` for use by downstream notebooks (harmonisation, EDA, modelling).

In [10]:
parquet_path = DATA_PROCESSED / "mi_adhd_feb2026.parquet"
mi_adhd.to_parquet(parquet_path, engine="pyarrow", compression="snappy", index=False)
print(f"Saved: {parquet_path.name}  ({parquet_path.stat().st_size / 1024:.1f} KB)")

Saved: mi_adhd_feb2026.parquet  (41.4 KB)
